### Diffusion Model - Soft + Hard Combined Data

Train the same partial-diffusion DDPM (`EnhancedDiffusionModel`) on the **unaugmented** soft+hard combined dataset from `prepare_soft_hard_combined.ipynb`.

#### Data & Scalers
- Train: `data/combined_soft_hard_train.xlsx`
- Val: `data/combined_soft_hard_val.xlsx`
- Scalers: `artifacts/high_stiffness/scaler_x_soft_hard.pkl`, `artifacts/high_stiffness/scaler_y_soft_hard.pkl`

#### Model columns (by name; ID columns excluded)
- Inputs (`X`): `Ep1`, `Ep2`, `Ep3`, `Epp1`, `Epp2`, `Epp3`, `t1`, `t2`
- Conditions (`y`): `EpH`, `EppH`
- Partial diffusion: only `t1`, `t2` are noised; moduli stay fixed

#### Outputs (new filenames under `data/`, do not overwrite previous artifacts)
- `artifacts/high_stiffness/diffusion_model_soft_hard_best.pt`
- `artifacts/high_stiffness/diffusion_model_soft_hard_final.pt`
- `artifacts/high_stiffness/diffusion_training_history_soft_hard.csv`
- `artifacts/high_stiffness/noise_prediction_data_soft_hard.pkl`
- `data/diffusion_loss_curves_soft_hard.png`
- `data/diffusion_t1_noise_scatter_soft_hard.png`
- `data/diffusion_t2_noise_scatter_soft_hard.png`
- `data/diffusion_feature_importance_*_soft_hard.png`
- `data/diffusion_condition_importance_soft_hard.png`

Architecture, schedule, and plotting style match `diffusion_model_new.ipynb`.


### Environment Settings

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
import math
import os

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from matplotlib.ticker import MultipleLocator

print("All modules imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Use serif + bold global settings before drawing
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

### 1. Diffusion Scheduler

In [ ]:
class DiffusionSchedule(nn.Module):
    def __init__(self, T: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02, eps: float = 1e-8):
        super().__init__()
        self.T = T
        self.eps = eps

        beta = torch.linspace(beta_start, beta_end, T)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]], dim=0)

        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x0[:, 6:])

        ab = self.alpha_bar.to(x0.device)[t].clamp(self.eps, 1.0 - self.eps)
        sqrt_ab = ab.sqrt().view(-1, 1)
        sqrt_omb = (1.0 - ab).sqrt().view(-1, 1)

        x_mod = x0[:, :6]  # Ep1-Epp3 remain unchanged
        x_dist = x0[:, 6:]  # t1, t2 undergo diffusion
        x_dist_t = sqrt_ab * x_dist + sqrt_omb * noise

        return torch.cat([x_mod, x_dist_t], dim=1)

print("Diffusion scheduler defined")

### 2. Time Embedding

In [ ]:
class TimeEmbedding(nn.Module):
    """Sinusoidal time embedding"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        half = dim // 2
        factor = math.log(10000.0) / (half - 1)
        self.register_buffer('factor', torch.tensor(factor))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        emb_scale = torch.exp(-self.factor * torch.arange(half, device=t.device))
        emb = t[:, None].float() * emb_scale[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=1)

print("Time embedding defined")

### 3. Optimized Diffusion Model (Enhanced Capacity + LayerNorm)

In [ ]:
class EnhancedDiffusionModel(nn.Module):
    """Enhanced diffusion model: deeper network with better regularization"""
    def __init__(self, x_dim=8, y_dim=2, time_dim=32, hidden_dim=256):
        super().__init__()
        self.x_dim = x_dim

        # === Moduli branch (Ep1-Epp3): enhanced capacity ===
        self.moduli_embed = nn.Sequential(
            nn.Linear(6, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Distribution branch (t1, t2): deep MLP ===
        self.dist_mlp = nn.Sequential(
            nn.Linear(2, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128)
        )

        # === Y conditioning branch (EpH, EppH) ===
        self.y_mlp = nn.Sequential(
            nn.Linear(y_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 128)
        )

        # === Time embedding branch ===
        self.time_embed = TimeEmbedding(time_dim)
        self.time_fc = nn.Sequential(
            nn.Linear(time_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 128)
        )

        # === Fusion layer: deep network ===
        fusion_dim = 128 + 128 + 128 + 128  # moduli + dist + y + time
        self.fusion_fc = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 2)  # Output noise prediction (t1, t2)
        )

    def forward(self, x, t, y):
        """Forward propagation
        x: [B, 8] - [Ep1, Ep2, Ep3, Epp1, Epp2, Epp3, t1_noisy, t2_noisy]
        t: [B] - timestep
        y: [B, 2] - condition [EpH, EppH]
        """
        x_mod = x[:, :6]   # Ep1-Epp3
        x_dist = x[:, 6:]  # t1, t2 (noisy)

        # Feature extraction from each branch
        mod_feat = self.moduli_embed(x_mod)
        dist_feat = self.dist_mlp(x_dist)
        y_feat = self.y_mlp(y)
        t_emb = self.time_embed(t)
        t_feat = self.time_fc(t_emb)

        # Fuse all features
        fused = torch.cat([mod_feat, dist_feat, y_feat, t_feat], dim=1)
        eps_pred = self.fusion_fc(fused)

        return eps_pred

print("Enhanced diffusion model defined")

### 4. Data Loading

In [ ]:
# Data paths
train_path = Path("data/combined_soft_hard_train.xlsx")
val_path = Path("data/combined_soft_hard_val.xlsx")
sx_path = Path("artifacts/high_stiffness/scaler_x_soft_hard.pkl")
sy_path = Path("artifacts/high_stiffness/scaler_y_soft_hard.pkl")

# Check file existence
if not train_path.exists():
    raise FileNotFoundError(f"{train_path} does not exist. Please run prepare_soft_hard_combined.ipynb first")
if not val_path.exists():
    raise FileNotFoundError(f"{val_path} does not exist. Please run prepare_soft_hard_combined.ipynb first")
if not (sx_path.exists() and sy_path.exists()):
    raise FileNotFoundError("scaler_x_soft_hard.pkl / scaler_y_soft_hard.pkl do not exist. Please run prepare_soft_hard_combined.ipynb first")

# 1) Load data
df_train = pd.read_excel(train_path, header=0)
df_val = pd.read_excel(val_path, header=0)

X_cols = ["Ep1", "Ep2", "Ep3", "Epp1", "Epp2", "Epp3", "t1", "t2"]
y_cols = ["EpH", "EppH"]

missing_tr = [c for c in X_cols + y_cols if c not in df_train.columns]
missing_va = [c for c in X_cols + y_cols if c not in df_val.columns]
if missing_tr or missing_va:
    raise ValueError(f"Missing columns — train: {missing_tr}, val: {missing_va}")

# Select by column name (exclude NewGroupID / HardSetID / MaterialType / Split)
X_train_raw = df_train[X_cols].to_numpy(dtype=float)
Y_train_raw = df_train[y_cols].to_numpy(dtype=float)
X_val_raw = df_val[X_cols].to_numpy(dtype=float)
Y_val_raw = df_val[y_cols].to_numpy(dtype=float)

print(f"[Data] Train set: {X_train_raw.shape}, Val set: {X_val_raw.shape}")
if "MaterialType" in df_train.columns:
    print("[Data] Train MaterialType counts:")
    print(df_train["MaterialType"].value_counts().to_string())
if "MaterialType" in df_val.columns:
    print("[Data] Val MaterialType counts:")
    print(df_val["MaterialType"].value_counts().to_string())

# 2) Load soft+hard scalers (fitted on combined training set only)
scaler_x = joblib.load(sx_path)
scaler_y = joblib.load(sy_path)
print(f"[Data] Loaded scaler_x_soft_hard.pkl & scaler_y_soft_hard.pkl")

# 3) Standardize
X_train = scaler_x.transform(X_train_raw)
Y_train = scaler_y.transform(Y_train_raw)
X_val = scaler_x.transform(X_val_raw)
Y_val = scaler_y.transform(Y_val_raw)

# 4) Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

x_train = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train = torch.tensor(Y_train, dtype=torch.float32, device=device)
x_val = torch.tensor(X_val, dtype=torch.float32, device=device)
y_val = torch.tensor(Y_val, dtype=torch.float32, device=device)

print(f"Train tensors: X={x_train.shape}, Y={y_train.shape}")
print(f"Val tensors: X={x_val.shape}, Y={y_val.shape}")


### 5. Model Initialization and Training Setup

In [ ]:
# Initialize components
schedule = DiffusionSchedule(T=1000).to(device)
model = EnhancedDiffusionModel(hidden_dim=384).to(device)  # Increased hidden_dim

# Optimizer: use AdamW
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# === EMA model (for stable validation) ===
ema_decay = 0.9999
ema_model = EnhancedDiffusionModel(hidden_dim=384).to(device)
ema_model.load_state_dict(model.state_dict())
for p in ema_model.parameters():
    p.requires_grad_(False)

# === Fixed validation t and noise (reduce validation metric variance) ===
seeds = [12345, 67890, 13579]

def make_gen(seed: int):
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    return g

t_val_fixed_list = [
    torch.randint(0, schedule.T, (x_val.size(0),), device=device, generator=make_gen(s))
    for s in seeds
]
noise_val_fixed_list = [
    torch.randn(x_val.size(0), 2, device=device, generator=make_gen(s*2 + 1))
    for s in seeds
]

# === Learning rate scheduler ===
# Warmup + Cosine Annealing
warmup_epochs = 20
scheduler_warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
scheduler_cosine = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=100, T_mult=2, eta_min=5e-6
)

# Data loader
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=256, shuffle=True)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Training setup complete!")

### 6. Training Main Loop

In [ ]:
# === Set random seeds for reproducibility ===
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Random seeds set for reproducibility")

# Training parameters
train_losses, val_losses = [], []
epochs = 800
patience = 250
best_val_loss = float('inf')
patience_counter = 0
min_delta = 1e-5

# === Data structures for saving training history ===
training_history = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'learning_rate': [],
    'best_val_loss': []
}

# === Data structures for saving noise predictions (for scatter plots) ===
noise_prediction_data = {
    'epoch': [],
    'true_noise_t1': [],
    'pred_noise_t1': [],
    'true_noise_t2': [],
    'pred_noise_t2': []
}

print("Starting training...")
print("="*80)

for epoch in range(1, epochs + 1):
    # === Training phase ===
    model.train()
    total_loss = 0.0
    
    for xb, yb in train_loader:
        # Random timestep and noise
        t = torch.randint(0, schedule.T, (xb.size(0),), device=device)
        noise_d = torch.randn(xb.size(0), 2, device=device)
        
        # Forward diffusion (add noise only to t1, t2)
        x_noisy = schedule.q_sample(xb, t, noise_d)
        
        # Predict noise
        eps_pred = model(x_noisy, t, yb)
        
        # Compute loss
        loss = F.mse_loss(eps_pred, noise_d)
        
        # Backpropagation
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Update EMA model
        with torch.no_grad():
            for p_ema, p in zip(ema_model.parameters(), model.parameters()):
                p_ema.mul_(ema_decay).add_(p.detach(), alpha=(1.0 - ema_decay))
        
        total_loss += loss.item() * xb.size(0)
    
    train_losses.append(total_loss / len(train_loader.dataset))
    
    # === Validation phase (using EMA model and fixed t/noise) ===
    ema_model.eval()
    with torch.no_grad():
        vloss_sum = 0.0
        for t_fix, n_fix in zip(t_val_fixed_list, noise_val_fixed_list):
            x_noisy_val = schedule.q_sample(x_val, t_fix, n_fix)
            eps_val = ema_model(x_noisy_val, t_fix, y_val)
            vloss_sum += F.mse_loss(eps_val, n_fix).item()
        val_losses.append(vloss_sum / len(t_val_fixed_list))
    
    # === Learning rate scheduling ===
    if epoch <= warmup_epochs:
        scheduler_warmup.step()
    else:
        scheduler_cosine.step(epoch - warmup_epochs)
    
    # === Early stopping check ===
    current_val_loss = val_losses[-1]
    
    if current_val_loss < best_val_loss - min_delta:
        best_val_loss = current_val_loss
        patience_counter = 0
        # Save best model
        Path("data").mkdir(parents=True, exist_ok=True)
        torch.save(ema_model.state_dict(), "artifacts/high_stiffness/diffusion_model_soft_hard_best.pt")
    else:
        patience_counter += 1
    
    # === Record training history ===
    lr_current = optimizer.param_groups[0]['lr']
    training_history['epoch'].append(epoch)
    training_history['train_loss'].append(train_losses[-1])
    training_history['val_loss'].append(val_losses[-1])
    training_history['learning_rate'].append(lr_current)
    training_history['best_val_loss'].append(best_val_loss)
    
    # === Periodically save noise predictions (every 50 epochs for scatter plots) ===
    if epoch % 50 == 0 or epoch == 1 or epoch == epochs:
        ema_model.eval()
        with torch.no_grad():
            # Use fixed validation data for prediction
            t_sample = t_val_fixed_list[0]
            noise_sample = noise_val_fixed_list[0]
            x_noisy_sample = schedule.q_sample(x_val, t_sample, noise_sample)
            eps_pred_sample = ema_model(x_noisy_sample, t_sample, y_val)
            
            # Convert to numpy arrays
            noise_true_np = noise_sample.cpu().numpy()
            eps_pred_np = eps_pred_sample.cpu().numpy()
            
            # Save to dictionary (only save first 1000 samples to save space)
            sample_size = min(1000, noise_true_np.shape[0])
            noise_prediction_data['epoch'].append(epoch)
            noise_prediction_data['true_noise_t1'].append(noise_true_np[:sample_size, 0])
            noise_prediction_data['pred_noise_t1'].append(eps_pred_np[:sample_size, 0])
            noise_prediction_data['true_noise_t2'].append(noise_true_np[:sample_size, 1])
            noise_prediction_data['pred_noise_t2'].append(eps_pred_np[:sample_size, 1])
    
    # Print progress
    if epoch % 10 == 0 or epoch == 1:
        loss_ratio = val_losses[-1] / train_losses[-1]
        print(f"Epoch {epoch:3d}/{epochs} | "
              f"Train: {train_losses[-1]:.6f} | "
              f"Val: {val_losses[-1]:.6f} | "
              f"Ratio: {loss_ratio:.3f} | "
              f"LR: {lr_current:.2e} | "
              f"Patience: {patience_counter}/{patience}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        print(f"Best validation loss: {best_val_loss:.6f}")
        break

# Save final model
Path("data").mkdir(parents=True, exist_ok=True)
torch.save(ema_model.state_dict(), "artifacts/high_stiffness/diffusion_model_soft_hard_final.pt")

# === Save training history to CSV file ===
history_df = pd.DataFrame(training_history)
history_df.to_csv('artifacts/high_stiffness/diffusion_training_history_soft_hard.csv', index=False)
print(f"\nTraining history saved to: artifacts/high_stiffness/diffusion_training_history_soft_hard.csv")

# === Save noise prediction data to pickle file ===
import pickle
with open('artifacts/high_stiffness/noise_prediction_data_soft_hard.pkl', 'wb') as f:
    pickle.dump(noise_prediction_data, f)
print(f"Noise prediction data saved to: artifacts/high_stiffness/noise_prediction_data_soft_hard.pkl")

print("\nTraining completed!")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Model saved to: artifacts/high_stiffness/diffusion_model_soft_hard_best.pt")
print(f"\nSaved data files:")
print(f"  - artifacts/high_stiffness/diffusion_training_history_soft_hard.csv (training history)")
print(f"  - artifacts/high_stiffness/noise_prediction_data_soft_hard.pkl (noise prediction data)")
print(f"  These data can be used to reproduce training results and plot loss curves/scatter plots")


### 7. Validation Set R² Evaluation

In [ ]:
# Load best model (suppress torch.load warning)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ema_model.load_state_dict(torch.load("artifacts/high_stiffness/diffusion_model_soft_hard_best.pt"))
ema_model.eval()

# Evaluate R² on validation set
with torch.no_grad():
    # Use fixed t and noise for prediction
    t_test = t_val_fixed_list[0]
    noise_test = noise_val_fixed_list[0]
    
    x_noisy_test = schedule.q_sample(x_val, t_test, noise_test)
    eps_pred = ema_model(x_noisy_test, t_test, y_val)
    
    # Compute R² (for noise prediction)
    eps_pred_np = eps_pred.cpu().numpy()
    noise_test_np = noise_test.cpu().numpy()
    
    r2_t1 = r2_score(noise_test_np[:, 0], eps_pred_np[:, 0])
    r2_t2 = r2_score(noise_test_np[:, 1], eps_pred_np[:, 1])
    r2_avg = (r2_t1 + r2_t2) / 2

print("="*60)
print("Validation Set R² Evaluation (Noise Prediction)")
print("="*60)
print(f"R² (t1 dimension): {r2_t1:.4f}")
print(f"R² (t2 dimension): {r2_t2:.4f}")
print(f"R² (average):      {r2_avg:.4f}")
print("="*60)

# MSE evaluation
mse = F.mse_loss(eps_pred, noise_test).item()
print(f"\nMSE (validation set): {mse:.6f}")


### 8. Training Summary


In [ ]:
print("\n" + "="*80)
print("Training Summary")
print("="*80)
print(f"Total training epochs: {len(train_losses)}")
print(f"Final training loss: {train_losses[-1]:.6f}")
print(f"Final validation loss: {val_losses[-1]:.6f}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Final loss ratio: {val_losses[-1]/train_losses[-1]:.3f}")
print(f"\nModel files:")
print(f"  - Best model: artifacts/high_stiffness/diffusion_model_soft_hard_best.pt")
print(f"  - Final model: artifacts/high_stiffness/diffusion_model_soft_hard_final.pt")
print(f"  - Loss curves: data/diffusion_loss_curves_soft_hard.png")
print(f"\nScaler files (used):")
print(f"  - {sx_path}")
print(f"  - {sy_path}")
print("="*80)


### 9. Reload training data and generate plots

**Important**: This section should be run after training is complete to visualize the results.

The training process saves all necessary data (training history and noise predictions) to files. To generate plots, simply run the cells below. This allows you to:
- Regenerate plots without retraining the model
- Customize plot parameters without affecting the saved data
- Generate plots multiple times as needed

In [ ]:
# Load training history from saved CSV file
import pandas as pd
import pickle
from pathlib import Path

results_dir = Path('data')
history_file = results_dir / 'diffusion_training_history_soft_hard.csv'
noise_file = results_dir / 'noise_prediction_data_soft_hard.pkl'

# Check if files exist
if not history_file.exists() or not noise_file.exists():
    print("=" * 70)
    print("ERROR: Required data files not found!")
    print("=" * 70)
    if not history_file.exists():
        print(f"Missing: {history_file}")
    if not noise_file.exists():
        print(f"Missing: {noise_file}")
    print("\nThese files are generated during training.")
    print("Please run the training cells first to train the model and save data.")
    print("=" * 70)
    raise FileNotFoundError("Run training cells first")

# Load training history
history_df = pd.read_csv(history_file)

# Extract data for plotting
train_losses = history_df['train_loss'].tolist()
val_losses = history_df['val_loss'].tolist()

print(f"Loaded training history: {len(history_df)} epochs")
print(f"Training completed from epoch 1 to {history_df['epoch'].max()}")


In [ ]:
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)
# Plot loss curves from loaded data
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.plot(train_losses, label='Train', linewidth=3)
plt.plot(val_losses, label='Validation', linewidth=3)
plt.xlabel('Epoch', fontsize=28, fontweight='bold', family='serif', labelpad=10)
plt.ylabel('MSE Loss', fontsize=28, fontweight='bold', family='serif', labelpad=5)
plt.xticks(fontsize=22, fontweight='bold', family='serif')
plt.yticks(fontsize=22, fontweight='bold', family='serif')
plt.legend(prop={'size': 24, 'weight': 'bold', 'family': 'serif'})
plt.grid(False)
plt.tight_layout()

ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(base=200))
ax.yaxis.set_major_locator(MultipleLocator(base=0.4))

for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.savefig('data/diffusion_loss_curves_soft_hard.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Loss curves saved to: data/diffusion_loss_curves_soft_hard.png")


In [ ]:
# Load noise prediction data
import numpy as np
from sklearn.metrics import r2_score
import warnings

with open(noise_file, 'rb') as f:
    noise_data = pickle.load(f)

# Get the last epoch's predictions
last_epoch_idx = -1
noise_true_t1 = noise_data['true_noise_t1'][last_epoch_idx]
noise_pred_t1 = noise_data['pred_noise_t1'][last_epoch_idx]
noise_true_t2 = noise_data['true_noise_t2'][last_epoch_idx]
noise_pred_t2 = noise_data['pred_noise_t2'][last_epoch_idx]

# ========== Re-evaluate on full validation set for accurate R² ==========
print("Re-evaluating on full validation set...")

# Load best model (suppress torch.load warning)
ema_model_eval = EnhancedDiffusionModel(hidden_dim=384).to(device)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ema_model_eval.load_state_dict(torch.load("artifacts/high_stiffness/diffusion_model_soft_hard_best.pt"))
ema_model_eval.eval()

# Use the same fixed validation data as training
with torch.no_grad():
    t_test = t_val_fixed_list[0]
    noise_test = noise_val_fixed_list[0]
    
    x_noisy_test = schedule.q_sample(x_val, t_test, noise_test)
    eps_pred = ema_model_eval(x_noisy_test, t_test, y_val)
    
    # Convert to numpy
    eps_pred_np = eps_pred.cpu().numpy()
    noise_test_np = noise_test.cpu().numpy()
    
    # Calculate R² on full validation set
    r2_t1 = r2_score(noise_test_np[:, 0], eps_pred_np[:, 0])
    r2_t2 = r2_score(noise_test_np[:, 1], eps_pred_np[:, 1])
    r2_avg = (r2_t1 + r2_t2) / 2

print(f"\n{'='*60}")
print(f"Official R² Scores (Full Validation Set)")
print(f"{'='*60}")
print(f"Validation samples: {noise_test_np.shape[0]}")
print(f"R² (t1): {r2_t1:.4f}")
print(f"R² (t2): {r2_t2:.4f}")
print(f"R² (avg): {r2_avg:.4f}")
print(f"{'='*60}")


In [ ]:
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)
# t1 dimension scatter plot
from matplotlib.ticker import MultipleLocator

# Use saved data for plotting (for visualization consistency)
x = noise_true_t1
y = noise_pred_t1

# Sample for display
np.random.seed(42)
max_points = 1500
if len(x) > max_points:
    idx = np.random.choice(len(x), max_points, replace=False)
    x_plot, y_plot = x[idx], y[idx]
    print(f"Sampled {max_points} points from {len(x)} points for display")
else:
    x_plot, y_plot = x, y

plt.figure(figsize=(8, 8))
plt.scatter(x_plot, y_plot, c='tab:blue', alpha=0.6, s=40, edgecolor='none')
lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
plt.plot([lo, hi], [lo, hi], '--', color='tab:red', linewidth=4)

# Use the full validation set R² for the plot
plt.text(0.05, 0.95, 
         f"R² = {r2_t1:.3f}",  
         transform=plt.gca().transAxes,
         fontsize=28, fontweight='bold', family='serif', 
         verticalalignment='top')

plt.xlabel("True Noise ($t_1$)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("Pred Noise ($t_1$)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=5)
plt.xticks(fontsize=22, fontweight='bold', family='serif')
plt.yticks(fontsize=22, fontweight='bold', family='serif')

ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(base=2))
ax.yaxis.set_major_locator(MultipleLocator(base=2))

for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.tight_layout()
plt.savefig('data/diffusion_t1_noise_scatter_soft_hard.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Note: Plot shows subset of data, but R² shown is from full validation set")


In [ ]:
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)
# t2 dimension scatter plot
x = noise_true_t2
y = noise_pred_t2

# Sample for display
np.random.seed(42)
max_points = 1500
if len(x) > max_points:
    idx = np.random.choice(len(x), max_points, replace=False)
    x_plot, y_plot = x[idx], y[idx]
    print(f"Sampled {max_points} points from {len(x)} points for display")
else:
    x_plot, y_plot = x, y

plt.figure(figsize=(8, 8))
plt.scatter(x_plot, y_plot, c='tab:orange', edgecolor='none', alpha=0.6, s=40)
lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
plt.plot([lo, hi], [lo, hi], '--', color='tab:blue', linewidth=4)

# Use the full validation set R² for the plot
plt.text(0.05, 0.95, 
         f"R² = {r2_t2:.3f}",  
         transform=plt.gca().transAxes,
         fontsize=28, fontweight='bold', family='serif', 
         verticalalignment='top')

plt.xlabel("True Noise ($t_2$)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("Pred Noise ($t_2$)", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=5)
plt.xticks(fontsize=22, fontweight='bold', family='serif')
plt.yticks(fontsize=22, fontweight='bold', family='serif')

ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(base=2))
ax.yaxis.set_major_locator(MultipleLocator(base=2))

for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.tight_layout()
plt.savefig('data/diffusion_t2_noise_scatter_soft_hard.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Note: Plot shows subset of data, but R² shown is from full validation set")

print("\n" + "="*70)
print("All plots generated successfully!")
print("="*70)
print(f"Official R² scores (full validation set, {noise_test_np.shape[0]} samples):")
print(f"  - R² (t1): {r2_t1:.4f}")
print(f"  - R² (t2): {r2_t2:.4f}")
print(f"  - R² (avg): {r2_avg:.4f}")
print("="*70)


### 10. Model Explainability Analysis

Analyze the importance of each feature on model prediction through Permutation Importance, understanding how the model utilizes different parameters for prediction.

In [ ]:
# Model Explainability Analysis: Permutation Importance
# Measure the importance of each feature by shuffling its values

# Analysis setup
ema_model.eval()
torch.manual_seed(0)
np.random.seed(0)

assert x_val.shape[1] == 8, "Expecting 8 feature columns."
feature_names = ['Ep1', 'Ep2', 'Ep3', 'Epp1', 'Epp2', 'Epp3', 't1', 't2']

# Select analysis subset to avoid excessive computation
SUBSET = min(4096, x_val.size(0))
idx_sub = torch.randperm(x_val.size(0), device=device)[:SUBSET]
x_eval = x_val[idx_sub].clone()
y_eval = y_val[idx_sub].clone()

print(f"Using {SUBSET} samples for explainability analysis")

# =========================================
# Evaluation Function: Compute noise prediction MSE
# =========================================
@torch.no_grad()
def eval_noise_mse(x_in, y_in, batch=1024, t_all=None, eps_all=None):
    """
    Evaluate model MSE on given inputs
    Args:
        x_in: Input features [N, 8]
        y_in: Conditions [N, 2] 
        t_all: Fixed timesteps [N] (optional)
        eps_all: Fixed noise [N, 2] (optional)
    Returns:
        mse_total: Overall MSE
        mse_dim: Dimension-wise MSE [2]
    """
    N = x_in.size(0)
    sum_dim = torch.zeros(2, device=x_in.device)
    n = 0
    
    for i in range(0, N, batch):
        xb = x_in[i:i+batch]
        yb = y_in[i:i+batch]
        b = xb.size(0)
        
        if t_all is None:
            tb = torch.randint(0, schedule.T, (b,), device=xb.device)
            eb = torch.randn(b, 2, device=xb.device)
        else:
            tb = t_all[i:i+batch]
            eb = eps_all[i:i+batch]
        
        x_noisy = schedule.q_sample(xb, tb, eb)
        pred = ema_model(x_noisy, tb, yb)
        
        loss_dim = (pred - eb) ** 2
        sum_dim += loss_dim.sum(0)
        n += b
    
    mse_dim = (sum_dim / n).detach().cpu().numpy()
    mse_total = float(mse_dim.mean())
    
    return mse_total, mse_dim

print("Evaluation function defined")

In [ ]:
# =========================================
# Permutation Importance Calculation
# =========================================
print("Starting Permutation Importance calculation...")

# Analysis parameters
BATCH = 2048
REPEATS = 5  # Number of repeats for stability

# Initialize importance arrays
imps_total = np.zeros(len(feature_names), dtype=np.float64)  # Overall importance
imps_t1 = np.zeros(len(feature_names), dtype=np.float64)     # t1 dimension importance  
imps_t2 = np.zeros(len(feature_names), dtype=np.float64)     # t2 dimension importance

for repeat in range(REPEATS):
    print(f"  Repeat {repeat + 1}/{REPEATS}")
    t_all = torch.randint(0, schedule.T, (x_eval.size(0),), device=device)
    eps_all = torch.randn(x_eval.size(0), 2, device=device)
    base_total, base_dim = eval_noise_mse(x_eval, y_eval, batch=BATCH, t_all=t_all, eps_all=eps_all)
    
    for j in range(x_eval.shape[1]):
        x_perm = x_eval.clone()
        perm_idx = torch.randperm(x_perm.size(0), device=device)
        x_perm[:, j] = x_perm[perm_idx, j]  
        tot, dim = eval_noise_mse(x_perm, y_eval, batch=BATCH, t_all=t_all, eps_all=eps_all)
        imps_total[j] += (tot - base_total)
        imps_t1[j] += (dim[0] - base_dim[0])
        imps_t2[j] += (dim[1] - base_dim[1])

imps_total /= REPEATS
imps_t1 /= REPEATS
imps_t2 /= REPEATS

print("Permutation Importance calculation completed!")
print(f"\nFeature importance results:")
for i, name in enumerate(feature_names):
    print(f"  {name}: Overall={imps_total[i]:.6f}, t1={imps_t1[i]:.6f}, t2={imps_t2[i]:.6f}")

In [ ]:
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)
# =========================================
# Visualize Feature Importance
# =========================================

def plot_ranked_bar(values, names, title):
    """Plot ranked feature importance bar chart"""
    # Label mapping: convert feature names to LaTeX format
    label_map = {
        'Ep1': r"$E_1^{\prime}$",
        'Ep2': r"$E_2^{\prime}$",
        'Ep3': r"$E_3^{\prime}$",
        'Epp1': r"$E_1^{\prime\prime}$",
        'Epp2': r"$E_2^{\prime\prime}$",
        'Epp3': r"$E_3^{\prime\prime}$",
        't1': r"$t_1$",
        't2': r"$t_2$"
    }
    
    # Handle negative values (occasional numerical noise)
    vals = np.maximum(values, 0)
    # Normalize to [0,1]
    vals = vals / (vals.max() + 1e-12) if vals.max() > 0 else vals
    # Sort by importance (descending)
    order = np.argsort(-vals)
    
    plt.figure(figsize=(6, 8))
    plt.bar(range(len(vals)), vals[order])
    
    # Apply label mapping to x-axis labels
    display_labels = [label_map.get(names[i], names[i]) for i in order]
    plt.xticks(range(len(vals)), display_labels, 
               rotation=45, fontweight='bold', fontsize=22, fontfamily='serif')
    plt.yticks(fontsize=22, fontweight='bold', family='serif')
    plt.ylabel("Normalized ΔMSE", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)

    # Thicken the axes spines
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    # Thicken the tick marks
    plt.tick_params(axis='both', which='major', length=8, width=3)
    
    plt.tight_layout()
    return order

print("Plotting feature importance charts...")

# 1. Overall importance
print("\n1. Overall Feature Importance:")
order_total = plot_ranked_bar(imps_total, feature_names, "Feature Importance Ranking (Overall)")
plt.savefig('data/diffusion_feature_importance_total_soft_hard.png', dpi=300, bbox_inches='tight')
plt.show()

for i, idx in enumerate(order_total):
    print(f"   {i+1}. {feature_names[idx]}: {imps_total[idx]:.6f}")

# 2. t1 dimension importance  
print("\n2. t1 Dimension Feature Importance:")
order_t1 = plot_ranked_bar(imps_t1, feature_names, "Feature Importance Ranking (t1 Dimension)")
plt.savefig('data/diffusion_feature_importance_t1_soft_hard.png', dpi=300, bbox_inches='tight')
plt.show()

for i, idx in enumerate(order_t1):
    print(f"   {i+1}. {feature_names[idx]}: {imps_t1[idx]:.6f}")

# 3. t2 dimension importance
print("\n3. t2 Dimension Feature Importance:")
order_t2 = plot_ranked_bar(imps_t2, feature_names, "Feature Importance Ranking (t2 Dimension)")
plt.savefig('data/diffusion_feature_importance_t2_soft_hard.png', dpi=300, bbox_inches='tight')
plt.show()

for i, idx in enumerate(order_t2):
    print(f"   {i+1}. {feature_names[idx]}: {imps_t2[idx]:.6f}")


In [ ]:
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)
# =========================================
# Condition (EpH, EppH) Importance Analysis
# =========================================
print("\n" + "="*60)
print("Condition Importance Analysis")
print("="*60)

# Test condition importance: compare normal vs shuffled conditions
torch.manual_seed(42)
t_test = torch.randint(0, schedule.T, (x_eval.size(0),), device=device)
eps_test = torch.randn(x_eval.size(0), 2, device=device)

# 1. Normal condition
y_intact = y_eval.clone()
intact_total, intact_dim = eval_noise_mse(x_eval, y_intact, batch=BATCH, t_all=t_test, eps_all=eps_test)

# 2. Shuffled condition  
y_shuffled = y_eval.clone()
shuffle_idx = torch.randperm(y_shuffled.size(0), device=device)
y_shuffled = y_shuffled[shuffle_idx]
shuffled_total, shuffled_dim = eval_noise_mse(x_eval, y_shuffled, batch=BATCH, t_all=t_test, eps_all=eps_test)

# Calculate relative increase
increase_total = (shuffled_total - intact_total) / intact_total * 100
increase_t1 = (shuffled_dim[0] - intact_dim[0]) / intact_dim[0] * 100  
increase_t2 = (shuffled_dim[1] - intact_dim[1]) / intact_dim[1] * 100

print(f"[Condition Intact] Overall MSE={intact_total:.6f}, t1={intact_dim[0]:.6f}, t2={intact_dim[1]:.6f}")
print(f"[Condition Shuffled] Overall MSE={shuffled_total:.6f}, t1={shuffled_dim[0]:.6f}, t2={shuffled_dim[1]:.6f}")
print(f"\nPerformance Degradation:")
print(f"  Overall: {increase_total:.1f}%")
print(f"  t1 dimension: {increase_t1:.1f}%") 
print(f"  t2 dimension: {increase_t2:.1f}%")

# Visualization comparison
fig, ax = plt.subplots(figsize=(10, 8))
categories = ['Overall', '$t_1$', '$t_2$']
intact_values = [intact_total, intact_dim[0], intact_dim[1]]
shuffled_values = [shuffled_total, shuffled_dim[0], shuffled_dim[1]]

x_pos = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x_pos - width/2, intact_values, width, label='Intact')
bars2 = ax.bar(x_pos + width/2, shuffled_values, width, label='Shuffled')

ax.set_xticks(x_pos)
ax.set_xticklabels(categories, fontsize=22, fontweight='bold', fontfamily='serif')
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels([f'{y:.2f}' for y in ax.get_yticks()], fontsize=22, fontweight='bold', fontfamily='serif')
ax.set_ylabel('MSE', fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
ax.legend(prop={'size':24,'weight':'bold','family':'serif'})

# Thicken axes spines
for spine in ax.spines.values():
    spine.set_linewidth(3)
# Thicken tick marks
ax.tick_params(axis='both', which='major', length=8, width=3)

plt.tight_layout()
plt.savefig('data/diffusion_condition_importance_soft_hard.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nConclusion:")
if increase_total > 100:
    print(f"Condition is extremely important! Performance degrades by {increase_total:.0f}% when shuffled")
elif increase_total > 50:
    print(f"Condition is important! Performance degrades by {increase_total:.0f}% when shuffled")
else:
    print(f"Condition has moderate importance, performance degrades by {increase_total:.0f}% when shuffled")


In [ ]:
# ============================
# Partial y-shuffle ablation
# Only one column of the conditional vector y (for example, column 0) is shuffled, and the rest of the columns remain unchanged
# ============================
def run_eval(y_in, repeats=REPEATS):
    totals, dims = [], []
    for _ in range(repeats):
        t_all   = torch.randint(0, schedule.T, (x_eval.size(0),), device=device)
        eps_all = torch.randn(x_eval.size(0), 2, device=device)
        tot, dim = eval_noise_mse(x_eval, y_in, batch=BATCH, t_all=t_all, eps_all=eps_all)
        totals.append(tot); dims.append(dim)
    return float(np.mean(totals)), np.mean(np.stack(dims,0),0)

# intact baseline
base_total, base_dim = run_eval(y_eval)

# partial shuffle
y_part = y_eval.clone()
perm_idx = torch.randperm(y_eval.size(0), device=device)
y_part[:,0] = y_part[perm_idx,0] 
part_total, part_dim = run_eval(y_part)

# fully shuffled
perm_idx = torch.randperm(y_eval.size(0), device=device)
y_full = y_eval[perm_idx]
full_total, full_dim = run_eval(y_full)

print("Baseline intact:", base_total, base_dim)
print("Partial shuffle:", part_total, part_dim)
print("Full shuffle:", full_total, full_dim)

labels = ["$t_1$", "$t_2$", "Overall"]
x = np.arange(len(labels))
width = 0.25

intact   = [base_dim[0], base_dim[1], base_total]
partial  = [part_dim[0], part_dim[1], part_total]
full     = [full_dim[0], full_dim[1], full_total]

fig, ax = plt.subplots(figsize=(10, 8))
ax.bar(x - width, intact,  width, label="Intact")
ax.bar(x,  partial, width, label="Partial shuffled")
ax.bar(x + width, full,   width, label="Full shuffled")

ax.set_ylabel("MSE", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
ax.set_xticks(x, labels, fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks([0, 0.4, 0.8, 1.2, 1.6],fontsize=22, fontweight='bold', family='serif')
ax.legend(loc='upper left', prop={'size':24,'weight':'bold','family':'serif'})

# Thicken the axes spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(3)
# Thicken the tick marks
plt.tick_params(axis='both', which='major', length=8, width=3)

fig.tight_layout()
plt.show()

In [ ]:
# =========================================
# Model Explainability Analysis Summary
# =========================================
print("\n" + "="*80)
print("Model Explainability Analysis Summary")
print("="*80)

print(f"\n  Analysis Configuration:")
print(f"  - Analysis samples: {SUBSET}")
print(f"  - Repeat times: {REPEATS}")
print(f"  - Model performance: R²={r2_avg:.4f}, MSE={best_val_loss:.6f}")

print(f"\n Feature Importance Ranking (Overall):")
for i, idx in enumerate(order_total[:5]):  # Show top 5 most important
    print(f"  {i+1}. {feature_names[idx]:>4s}: {imps_total[idx]:>8.4f} (ΔMSE)")

print(f"\n Key Findings:")
# Analyze most important feature types
ep_importance = np.sum([imps_total[i] for i in range(6)])  # Ep1-Epp3
t_importance = np.sum([imps_total[i] for i in range(6, 8)])  # t1, t2

print(f"  - Material parameters (Ep1-Epp3) total importance: {ep_importance:.4f}")
print(f"  - Distribution parameters (t1-t2) total importance: {t_importance:.4f}")

if ep_importance > t_importance:
    print(f"Material parameters are more important than distribution parameters ({ep_importance/t_importance:.2f}x)")
else:
    print(f"Distribution parameters are more important than material parameters ({t_importance/ep_importance:.2f}x)")

print(f"\n Condition (EpH, EppH) Importance:")
print(f"  - Performance degradation: {increase_total:.1f}% (when condition shuffled)")
if increase_total > 100:
    print(f"Condition is extremely important! Model heavily relies on target values")
else:
    print(f"Condition is important, but model has some generalization ability")

print(f"\n Saved Files:")
saved_files = [
    'data/diffusion_feature_importance_total_soft_hard.png',
    'data/diffusion_feature_importance_t1_soft_hard.png', 
    'data/diffusion_feature_importance_t2_soft_hard.png',
    'data/diffusion_condition_importance_soft_hard.png'
]
for file in saved_files:
    print(f" - {file}")

print(f"\n Model Interpretation:")
print(f"  Your diffusion model (R²={r2_avg:.3f}) successfully learned:")
print(f"  1. Relationships between material and distribution parameters")
print(f"  2. Guidance role of conditions in noise prediction")  
print(f"  3. Feature sensitivity across different dimensions (t1 vs t2)")

print("\n" + "="*80)
print("Model Explainability Analysis Completed!")
